### **Reading Food Hygiene Data Analysis and Visualisations**

### Questions we are looking to answer

1. **Target Volume:** How many food businesses in Reading are compliant versus non-compliant ones (ratings 0-2)?
2. **Rating Breakdown:** Within the non-compliant businesses, what is the exact distribution of each rating (0, 1, 2)?
3. **Struggling Businesses:** Is there a specific type of business (such as takeaways, restaurants, pubs or retail) that account for the lowest ratings?
4. **Geographic Distribution** Are low-rated businesses situated in specific areas across reading?

### Planned Steps
1. Load cleaned Reading FSA data from cleaning `jupyter_notebooks\reading_food_hygiene_prep.ipynb' - Verify load.
2. Analyse overall compliance volume (Target Volume) - Group together 0-2 ratings, and 3-5 ratings to calculate totals and percentages.
3. Break down non-compliant ratings (Rating Breakdown) - Filter the non-compliant businesses and compute the exact counts and frequencies for ratings 0, 1, and 2.
4. Investigate struggling Business Types (Struggling Businesses) -  Check which category of business have the worst ratings.
5. Investigate location of failing businesses (Geographic Distribution) - Check which areas in reading have the most low rated businesses.
6. Create a targeted list of failing businesses to support out conclusions (documented in our README.md file)


### Input (Data Source)
* **Data Source:** Cleaned Reading FSA data from `data/rfh_clean`

### Outputs
**Summary Metrics**
- Clean summary tables containing counts, percentages, and breakdowns for compliance, ratings, business types, and postcode districts.

**Visualisations**
- Compliance proportion chart (compliance vs non-compliant)
- A breakdown bar chart for ratings 0, 1 and 2
- A categorical comparison chart for struggling business types
- A postcode chart showing where low-rated business are clustered


---
### Setup
Install and Import required libraries

In [ ]:
%pip install matplotlib numpy pandas seaborn

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

print("Libraries imported successfully!")

---
### 1. Load cleaned Reading FSA dataset - Verify load

Load the cleaned Reading Food hygiene data, cleaning was carried out in `jupyter_notebooks\reading_food_hygiene_prep.ipynb`.

Cleaned data file is stored in the data directory `data/rfh_clean.csv`.
Verify load with a short message and display the head

In [ ]:
# Load the cleaned data file
df = pd.read_csv("../data/rfh_clean.csv")
print("Dataset loaded successfully!")

# Display the first few rows of the dataset and it's info
display(df.head())
display(df.info())

---
### 2. Analyse overall compliance volume (Target Volume)
Overall there are 1320 Businesses in the Reading dataset - we want to find out how many of them are Compliant and Non-compliant.
- Create a compliance column to state if the ratings fall into one of 2 categories, 3-5 (Compliant) and 0-2 (Non-compliant).
- Calculate counts and percentages for each and show them in one table.
- Visualise our findings using the most appropriate chart.

**Compliance Column**

In [ ]:
# Create compliance column based on hygiene score
df['Compliance_Status'] = df['RatingValue'].apply(lambda x: 'Compliant' if x >= 3 else 'Non-Compliant')

# Verify the new column has been added correctly
display(df.head())

**Calculate Counts & Percentages**

In [ ]:
# Display the counts of each compliance status and their percentages
compliance_counts = df['Compliance_Status'].value_counts()
compliance_percentages = df['Compliance_Status'].value_counts(normalize=True) * 100

compliance_summary = pd.DataFrame({
    'Count': compliance_counts,
    'Percentage': compliance_percentages.round(2)
})
display(compliance_summary)

**Visualise**

For this information I have chosen a simple **pie chart** using **matplotlib** to clearly illustrate the split between businesses meeting the Food Standards Agency hygiene standards and those that do not, making the overall compliance rate instantly clear at a glance.

### Conclusion

It immediately highlights that there is a vast majority of compliant businesses against a much smaller percentage of non compliant establishments.

In [ ]:
colours = ['#2ca02c', '#d62728']  # Green for Compliant, Red for Non-Compliant
plt.pie(compliance_counts, labels=None, autopct='%1.1f%%', colors=colours)
plt.title('FSA Hygiene Compliance Rates in \nReading UK', fontweight='bold')
plt.legend(compliance_counts.index, loc='lower center', bbox_to_anchor=(0.5, -0.1))
plt.figtext(0.5, 0.01, 'Data Source: Food Standards Agency (FSA) - August 2026', ha='center', fontsize=8)
plt.show()
   

---
### 3. Break down non-compliant ratings (Rating Breakdown)
This will show us how many food businesses have achieved a non-compliant score of 0-2 in Reading.
- Filter the dataset to just the non-compliant businesses in Reading.
- Calculate the number of businesses that have receieved each of the scores of 0, 1, and 2.
- Visualise our findings using the most appropriate chart.

**Filter the dataset to just the non-compliant businesses in Reading**

In [ ]:
# Filter the dataset to just non-compliant busninesses
non_compliant_df = df[df['Compliance_Status'] == 'Non-Compliant']
display(non_compliant_df.head())

**Calculate the number of businesses that have receieved each of the scores of 0, 1, and 2**

In [ ]:
rating_counts = non_compliant_df['RatingValue'].value_counts().sort_index()
display(rating_counts)

**Visualise**

For this information, I have chosen a **bar chart** using **Matplotlib** and **Seaborn** to clearly show the specific distribution of the hygiene ratings of 0, 1, and 2 across the non-compliant food businesses in Reading, making it easy to compare the number of each failing score at a glance.


### Conclusion

It immediately shows that there are a very low number of businesses that have failed to meet the FSA standards scoring zero (5), and there is a similarity in the number of businesses that scored 1 and 2 (41 and 39 respectively). 

In [ ]:
plt.figure(figsize=(8, 6))
sns.set_palette(palette='bright')
sns.barplot(x=rating_counts.index, y=rating_counts.values)
plt.title('Breakdown of Non-Compliant Businesses in Reading by Hygiene Rating', fontweight='bold')
plt.xlabel('Hygiene Rating', fontweight='bold')
plt.ylabel('Number of Non-compliant Businesses', fontweight='bold')
plt.figtext(0.5, 0.01, 'Data Source: Food Standards Agency (FSA) - August 2026', ha='center', fontsize=8)
plt.show()

---
### 4. Investigate struggling Business Types (Struggling Businesses)
Here we will now investigate which category of food businesses have the worst ratings in the Reading area out of the 85 we already know are struggling.
- **Group and count by business type:** Use the `non_compliant_df` dataframe we created earlier to focus on the non-compliant businesses to group and count by business type, to see which cetegories have the most failing scores.
- **Sort the results:** Order the counts from highest to lowest, so we can identify which business types are struggling the most.
- **Calculate the percentage:** Of each business type that make up the total of the 85 non-compliant establishments, rounding them to two decimal places.
- **Visualise our findings** using the most appropriate chart.

**Group and count by business type** and **Sort the results** highest to lowest.

In [ ]:
# Group the non-compliant businesses by business type and count them
struggling_businesses_by_type = non_compliant_df['BusinessType'].value_counts().sort_values(ascending=False)
percentages = (struggling_businesses_by_type / struggling_businesses_by_type.sum()) * 100

# Combine them in a single dataframe for better visualisation
struggling_businesses_df = pd.DataFrame({
    'Count': struggling_businesses_by_type,
    'Percentage': percentages.round(2)
})

display(struggling_businesses_df)

**Visualise**

For this information, I have chosen a **lolipop chart** using **MatplotLib** to clearly illistrate the breakdown of non-compliant businesses in Reading.

By using horizontal lines, paired with marker dots - having also sorted the categories into descending order (with the highest values at the top), it shows the exact distribution of non-compliant businesses across each category.


### Conclusion

It reveals in a non-fussy manner, that the three top catgories of *'struggling food businesses in Reading'* are in the sectors of:

***Retailers - other**

***Takeaway/sandwich shop**

***Restaurant/Cafe/Canteen**

In [ ]:
plt.figure(figsize=(10, 4))
plt.hlines(y=struggling_businesses_df.index, xmin=0, xmax=struggling_businesses_df['Count'], linewidth=5)
plt.plot(struggling_businesses_df['Count'], struggling_businesses_df.index, 'o', markersize=10, color='orange')
plt.gca().invert_yaxis()
plt.title('Non-compliant Food Businesses in Reading by Business Type', fontweight='bold')
plt.xlabel('Number of Non-compliant Businesses', fontweight='bold', color='blue')
plt.ylabel('Business Type', fontweight='bold', color='blue')

plt.subplots_adjust(bottom=0.2)  # adjust the bottom margin to make space
plt.figtext(0.5, 0.01, 'Data Source: Food Standards Agency (FSA) - August 2026', ha='center', fontsize=8)

plt.show()

Exploring and visualising this in a slightly different manner - we could also investigate which types of business are struggling and the severity of failing scores simultaneously.
In order to do this we could use **Seaborn** to create a **Heatmap** to clearly show the specific distribution of non-compliant business types in Reading, UK.
This would then show which business types accumilate the highest volume of non-compliance, while also highlighting whether those failures are in the most severe category (0 rartings), or spread across scores of 1 and 2.

In [ ]:
# Group business types by their hygiene ratings (0, 1, and 2)
heatmap_data = pd.crosstab(non_compliant_df['BusinessType'], non_compliant_df['RatingValue'])

# Create the heatmap
plt.figure(figsize=(10,6))
sns.heatmap(heatmap_data, annot=True, cmap='Blues', fmt='d', linecolor='black', linewidths=0.5)

# Labelling
plt.title('Heatmap of Non-Compliant Business, by Business Type and Hygiene Rating', fontsize=14, fontweight='bold', loc='right', color='navy')
plt.xlabel('Hygiene Rating Score', fontweight='bold', color='navy')
plt.ylabel('Business Type', fontweight='bold', color='navy')

# Force Matplotlib to show the bottom and right borders of the heatmap
for spine in plt.gca().spines.values():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(1)

plt.subplots_adjust(bottom=0.15)  # adjust the bottom margin to make space
plt.figtext(0.5, 0.01, 'Data Source: Food Standards Agency (FSA) - August 2026', ha='left', fontsize=8)


plt.show()

### Conclusion
From our heatmap, we can see that:

**Retailers - other**

Not only has it got the highest overall volume of non-compliant businesses (with 32 in total), but we can also see that the majority of their failures fall into the **1 rating**. It further shows this category also has the highest number of **0 Rating** Businesses - this demonstrates the business type and severity which would be of the biggest concern.

**Takeaway/sandwich shop** - **Restaurant/Cafe/Canteen**

These business types show high overall volumes of non-compliance, clustered primarily in the **1 and 2 Ratings**, this also highlights a widespread issue with these specific business types.


---
### 5. Investigate location of failing businesses (Geographic Distribution)
Investigate which areas in Reading have the most low rated businesses using a district . Is there a correlation to where the highest number of businesses are located?

Steps we will take to establish this:
- Get a unique list of Postcode Districts in Reading (Using the `PostCodeDistrict` in our dataset we created while cleaning), and sort them
- Establish where each of the the postcode districs are
- Break down how many failing businesses are there in each area and count them
- Calculate the percentage of failing busineesses in each district
- Visualise our findings using the most appropriate chart

**Get a unique list of Postcode Districts in Reading**

(Using the `PostCodeDistrict` in our dataset we created while cleaning), and sort them

In [ ]:
unique_districts = sorted(non_compliant_df['PostCodeDistrict'].unique())
unique_districts

**Establish where the areas are**

As there are only six postcodes in the area, we can just manually identify where they are using: https://en.wikipedia.org/wiki/RG_postcode_area

| PostCodeDistrict | Area |
| :--- | :--- |
| RG1 | Katesgrove, Newtown, Reading (central) |
| RG2 | Madejski Stadium, Whitley, Shinfield, Arborfield Garrison, Arborfield, Green Park Village |
| RG4 | Caversham, Sonning, Sonning Common, Kidmore End, Sonning Eye, Dunsden Green, Mapledurham, Chazey Heath, Tokers Green, Chalkhouse Green, Cane End |
| RG6 | Earley |
| RG30 | Tilehurst (east), Prospect Park, Burghfield village, West Reading, Southcote |
| RG31 | Calcot, Tilehurst (west) |

**Break down how many failing businesses are there in each area and count them**

In [ ]:
# Filtering the non-compliant businesses to only those with failing hygiene ratings (0, 1, and 2)
failing_df = non_compliant_df[non_compliant_df['RatingValue'].isin([0, 1, 2])]

# Crosstab to show the number of failing businesses in each district by Rating
district_breakdown = pd.crosstab(failing_df['PostCodeDistrict'], failing_df['RatingValue'])

# Total the failed businesses in each district
district_breakdown['Total'] = district_breakdown.sum(axis=1)
district_breakdown

**Compare the number of failing businesses, against the total number of food businesses in each district**

In [ ]:
# Count the total number of businesses in Reading
total_businesses = df['PostCodeDistrict'].value_counts()

# Add to the district breakdown dataframe
district_breakdown['Total_Food_Businesses'] = total_businesses

# Calculate the percentage of failing businesses in each district
district_breakdown['Failure_Rate_Percentage'] = (district_breakdown['Total'] / district_breakdown['Total_Food_Businesses'] * 100).round(2)

district_breakdown

**Visualise**

For this information, I have chosen two bar charts - one of which shows a direct comparison between the total number of food businesses in Reading, and the total number of non-compliant businesses, across each district. The second chart shows the level of compliance in each postcode district, broken down by hygiene rating. 

This was done to give a complete overview of the level of compliance in each area.

In [ ]:
# Show charts side by side using subplots
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# (chart 1) Compliant/ non-compliant businesses in each district
comparison_df = district_breakdown.reset_index().melt(id_vars='PostCodeDistrict', value_vars=['Total_Food_Businesses', 'Total'], var_name='Business_Type', value_name='Count')
sns.barplot(data=comparison_df, x='PostCodeDistrict', y='Count', hue='Business_Type', palette='bright', ax=axes[0])
max_count = comparison_df['Count'].max()
axes[0].set_yticks(np.arange(0, max_count + 25, 25))

# Labelling
axes[0].set_xlabel('Post Code District', fontweight='bold')
axes[0].set_ylabel('Number of Businesses', fontweight='bold')
axes[0].set_title('Comparison of Compliant and Non-Compliant \nBusinesseses by District in Reading, UK', fontweight='bold')

# Update the legend for clarity
legend_0 = axes[0].get_legend()
if legend_0:
    legend_0.set_title('Status')
    legend_0.texts[0].set_text('Compliant')
    legend_0.texts[1].set_text('Non-Compliant')

# (chart 2) Failing businesses in each district
sns.countplot(data=failing_df, x='PostCodeDistrict', hue='RatingValue', palette='bright', ax=axes[1])

# Update the legend for clarity
legend_1 = axes[1].get_legend()
if legend_1:
    legend_1.set_title('Rating Value')

# Labelling
axes[1].set_title('Breakdown of FSA Ratings in Reading UK \nfor Failing Businesses (grouped by Postcode District)', fontweight='bold')
axes[1].set_ylabel('Number of Businesses', fontweight='bold')
axes[1].set_xlabel('Postcode District', fontweight='bold')


plt.show()


### Conclusion

**Chart 1 - (Total of Business Vs. Non-compliant by District)**

This chart shows the scale of the problem of food businesses in Reading. While the number of failing businesses appear to be low in every district against the number of businesses that exist, it does show there are still failures in all of them. The highest concentration of which being the RG1 area, which from our earlier investigation we can see are **Katesgrove, Newtown, Reading (central)**.

**Chart 2 - Failing Businesses Grouped by Rating**

This chart shows the severity of the risk of the non-compliant businesses in each of the districts of Reading. Instead of lumping all the figures together, it clearly highlights how bad the level of non-compliance is in every district. The highest of which, is once again in the RG1 area **Katesgrove, Newtown, Reading (central)**, followed by RG30 and RG2 respectively.

**Both Charts**

Looking at both charts together like this we can see the largest number of failing food businesses in Reading as a whole, are situated in **Katesgrove, Newtown, Reading (central)**.

---
### 6. Create a targeted list of failing businesses in RG1 to support out conclusions

Extract and isolate details for specific non compliant businesses within the high priority **RG1** district to provide an actionable list for the sales team of the business.

**Conclusions:** A complete summary of the findings , regional comparisons, and final recommendations can be found in the `README.md` file located in the root directory.

In [282]:
target_list = non_compliant_df[non_compliant_df['PostCodeDistrict'] == 'RG1'][['BusinessName', 'AddressLine1', 'PostCode', 'RatingValue', 'PostCodeDistrict', 'BusinessType']].sort_values(by='RatingValue', ascending=True)
target_list

,BusinessName,AddressLine1,PostCode,RatingValue,PostCodeDistrict,BusinessType
533,JieLi Restaurant,19-23 Kings Street,RG1 2HG,0,RG1,Other catering premises
567,Kings Food and Wine,91-93 Kings Road,RG1 3DD,0,RG1,Retailers - other
952,Shirdi Saibaba Temple & Community,44 West Street,RG1 1TZ,0,RG1,Other catering premises
1219,Turkish Halal Food Centre,133-135 Oxford Road,RG1 7UU,0,RG1,Retailers - other
247,Chillim,56 St Mary's Butts,RG1 2LG,1,RG1,Restaurant/Cafe/Canteen
414,Fresh Gurkha Grocery,253 London Road,RG1 3NY,1,RG1,Retailers - other
107,Bierhaus,8 Queens Walk,RG1 7QF,1,RG1,Restaurant/Cafe/Canteen
8,7 Seas Fresh Fish,88 Oxford Road,RG1 7LJ,1,RG1,Retailers - other
452,Grape Tree,64 Broad Street Mall,RG1 7QE,1,RG1,Retailers - other
805,Paya/Korean Krispy,84 London Street,RG1 4SJ,1,RG1,Takeaway/sandwich shop


**Extract and save `Target List` to the `Data` folder**

In [283]:
# Save the Target list to a CSV file
target_list.to_csv('../data/target_list_RG1.csv', index=False)
print("Target list for RG1 saved to CSV successfully!")

Target list for RG1 saved to CSV successfully!
